# 02. Modele parametryczne i Bayes

Notebook obejmuje modele AFT, model zredukowany, diagnostykę graficzną oraz demonstracyjny bayesowski model Weibulla z MCMC.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, NelsonAalenFitter, WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter, ExponentialFitter, CoxPHFitter
from lifelines.statistics import logrank_test, proportional_hazard_test

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "Data" / "heart_failure_clinical_records_dataset.csv"
WORKSPACE = BASE_DIR
TABLES = WORKSPACE / "Tables"
FIGS = WORKSPACE / "Wykresy"

df = pd.read_csv(DATA_PATH).drop_duplicates()
df["duration"] = df["time"].astype(float)
df["event"] = df["DEATH_EVENT"].astype(int)
df["ef_low"] = (df["ejection_fraction"] < 35).astype(int)
df["creatinine_high"] = (df["serum_creatinine"] > 1.5).astype(int)
df["sodium_low"] = (df["serum_sodium"] < 135).astype(int)
df["log_cpk"] = np.log1p(df["creatinine_phosphokinase"])
df.head()

In [ ]:
cov_full=['age','ejection_fraction','serum_creatinine','serum_sodium','anaemia','diabetes','high_blood_pressure','sex','smoking','log_cpk']
cov_red=['age','anaemia','ejection_fraction','high_blood_pressure','serum_creatinine','serum_sodium']
full=df[['duration','event']+cov_full]
red=df[['duration','event']+cov_red]
models={
 'Weibull pełny': WeibullAFTFitter(penalizer=0.01).fit(full,'duration','event'),
 'Log-normal': LogNormalAFTFitter(penalizer=0.01).fit(full,'duration','event'),
 'Log-logistic': LogLogisticAFTFitter(penalizer=0.01).fit(full,'duration','event'),
 'Weibull zredukowany': WeibullAFTFitter(penalizer=0.01).fit(red,'duration','event')
}
pd.concat({k:m.summary[['coef','exp(coef)','p']] for k,m in models.items()})

In [ ]:
# Demonstracyjny model bayesowski jest estymowany w skrypcie przez algorytm Metropolisa-Hastingsa.
# W prezentacji raportowane są posterior summary i traceploty zapisane jako tabele/wykresy.

Model bayesowski ma ograniczony zakres: obejmuje wiek, frakcję wyrzutową i kreatyninę. Jego celem jest pokazanie realnie oszacowanego rozszerzenia, a nie zastąpienie modeli klasycznych.